# Feature Selection & Correlation

This notebook loads the raw processed data, cleans it, splits it into train/validation/test, and performs correlation-based feature selection. The final selected feature set (26 features) is saved to disk for use in the modeling notebook (`02_random_forest_model.ipynb`).

In [36]:
import json
import pandas as pd
import numpy as np

In [37]:
df = pd.read_csv(
   "../../data/processed/store_day.csv"
)
df.head()

,Location,Date,Gross_Sales,Discounts,Net_Sales,Tax,Total_Collected,Gross_Profit,Total_Cost,Transactions,...,Net_Sales_Lag_7,Transactions_Lag_1,Transactions_Lag_7,Profit_Rolling_7D,Net_Sales_Rolling_7D,Transactions_Rolling_7D,Profit_Growth_7D,Net_Sales_Growth_7D,Transactions_Growth_7D,Future_7D_Gross_Profit
0,Store 01 - Austin,2024-01-01,456.0,-37.15,418.85,34.56,453.41,212.82,206.03,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,397.297143
1,Store 01 - Austin,2024-01-02,238.0,-5.90,232.10,19.15,251.25,130.46,101.64,4.0,...,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,427.541429
2,Store 01 - Austin,2024-01-03,296.0,-40.72,255.28,21.07,276.35,136.06,119.22,2.0,...,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,476.461429
3,Store 01 - Austin,2024-01-04,478.0,-12.54,465.46,38.42,503.88,281.20,184.26,9.0,...,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,536.354286
4,Store 01 - Austin,2024-01-05,902.0,-59.07,842.93,69.54,912.47,487.54,355.39,16.0,...,NaN,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,495.822857


In [38]:
df.columns

Index(['Location', 'Date', 'Gross_Sales', 'Discounts', 'Net_Sales', 'Tax',
       'Total_Collected', 'Gross_Profit', 'Total_Cost', 'Transactions',
       'Units', 'Refund_Lines', 'Discounted_Lines',
       'Identified_Customer_Lines', 'Unique_SKUs', 'Store_Closed',
       'Refund_Amount', 'Weekday', 'Event', 'Season_Month_Event_Mult',
       'DOW_Mult', 'Month_Mult', 'Event_Mult', 'Discount_Line_Prob',
       'Refund_Txn_Prob', 'Profit_Margin', 'Discount_Rate',
       'Avg_Transaction_Value', 'Units_Per_Transaction', 'Profit_Lag_1',
       'Profit_Lag_7', 'Net_Sales_Lag_1', 'Net_Sales_Lag_7',
       'Transactions_Lag_1', 'Transactions_Lag_7', 'Profit_Rolling_7D',
       'Net_Sales_Rolling_7D', 'Transactions_Rolling_7D', 'Profit_Growth_7D',
       'Net_Sales_Growth_7D', 'Transactions_Growth_7D',
       'Future_7D_Gross_Profit'],
      dtype='object')

In [39]:
model_data = df.copy()

model_data = model_data.dropna(
    subset=["Future_7D_Gross_Profit"]
)

model_data.shape

(21720, 42)

In [40]:
closed_day_cols = [
    "Profit_Margin",
    "Discount_Rate",
    "Avg_Transaction_Value",
    "Units_Per_Transaction",
    "Profit_Growth_7D",
    "Net_Sales_Growth_7D",
    "Transactions_Growth_7D"
]

model_data.loc[
    model_data["Store_Closed"] == 1,
    closed_day_cols
] = 0

In [41]:
growth_cols = [
    "Profit_Growth_7D",
    "Net_Sales_Growth_7D",
    "Transactions_Growth_7D"
]

model_data[growth_cols] = model_data[growth_cols].fillna(0)

In [42]:
model_data.isna().sum()[model_data.isna().sum() > 0]

Refund_Amount               38
Profit_Lag_1                30
Profit_Lag_7               210
Net_Sales_Lag_1             30
Net_Sales_Lag_7            210
Transactions_Lag_1          30
Transactions_Lag_7         210
Profit_Rolling_7D          180
Net_Sales_Rolling_7D       180
Transactions_Rolling_7D    180
dtype: int64

In [43]:
model_data = (
    model_data
    .sort_values(["Location", "Date"])
    .groupby("Location")
    .apply(lambda x: x.iloc[7:])
    .reset_index(drop=True)
)

C:\Users\ShehabYousef\AppData\Local\Temp\ipykernel_28016\1452950607.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.iloc[7:])


In [44]:
train = model_data[model_data["Date"] <= "2025-06-30"].copy()

validation = model_data[
    (model_data["Date"] >= "2025-07-01") &
    (model_data["Date"] <= "2025-09-30")
].copy()

test = model_data[model_data["Date"] >= "2025-10-01"].copy()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (16200, 42)
Validation: (2760, 42)
Test: (2550, 42)


In [45]:
model_data["Refund_Amount"] = model_data["Refund_Amount"].fillna(0)


In [46]:
model_data.isna().sum()[model_data.isna().sum() > 0]

Series([], dtype: int64)

In [47]:
target = "Future_7D_Gross_Profit"

X_train = train.drop(columns=[target])
y_train = train[target]

X_validation = validation.drop(columns=[target])
y_validation = validation[target]

X_test = test.drop(columns=[target])
y_test = test[target]

## Feature Selection

In [48]:
feature_candidates = [
    "Gross_Sales",
    "Discounts",
    "Net_Sales",
    "Tax",
    "Total_Collected",
    "Gross_Profit",
    "Total_Cost",
    "Transactions",
    "Units",
    "Refund_Lines",
    "Discounted_Lines",
    "Identified_Customer_Lines",
    "Unique_SKUs",
    "Store_Closed",
    "Profit_Margin",
    "Discount_Rate",
    "Avg_Transaction_Value",
    "Units_Per_Transaction",
    "Profit_Lag_1",
    "Profit_Lag_7",
    "Net_Sales_Lag_1",
    "Net_Sales_Lag_7",
    "Transactions_Lag_1",
    "Transactions_Lag_7",
    "Profit_Rolling_7D",
    "Net_Sales_Rolling_7D",
    "Transactions_Rolling_7D",
    "Profit_Growth_7D",
    "Net_Sales_Growth_7D",
    "Transactions_Growth_7D"
]

X_train_candidates = X_train[feature_candidates]
X_validation_candidates = X_validation[feature_candidates]
X_test_candidates = X_test[feature_candidates]

In [49]:
correlation = (
    X_train_candidates
    .corrwith(y_train)
    .sort_values(key=abs, ascending=False)
)

correlation

Profit_Rolling_7D            0.929348
Net_Sales_Rolling_7D         0.922114
Transactions_Rolling_7D      0.912279
Gross_Profit                 0.743672
Profit_Lag_1                 0.740747
Unique_SKUs                  0.739779
Transactions                 0.739445
Tax                          0.733964
Total_Collected              0.733961
Net_Sales                    0.733960
Transactions_Lag_1           0.733446
Units                        0.731733
Net_Sales_Lag_1              0.730244
Gross_Sales                  0.723683
Total_Cost                   0.717699
Profit_Lag_7                 0.716949
Transactions_Lag_7           0.706521
Net_Sales_Lag_7              0.705244
Identified_Customer_Lines    0.668337
Discounted_Lines             0.600070
Discounts                   -0.439908
Refund_Lines                 0.261907
Profit_Margin               -0.063884
Profit_Growth_7D            -0.035951
Discount_Rate                0.033449
Store_Closed                 0.033004
Net_Sales_Gr

In [50]:
corr_matrix = X_train_candidates.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_pairs = (
    upper
    .stack()
    .sort_values(ascending=False)
)

high_corr_pairs[high_corr_pairs >= 0.90]

Net_Sales             Total_Collected              1.000000
Tax                   Total_Collected              1.000000
Net_Sales             Tax                          1.000000
Profit_Rolling_7D     Net_Sales_Rolling_7D         0.998731
Gross_Sales           Total_Cost                   0.998504
                      Net_Sales                    0.998066
                      Total_Collected              0.998066
                      Tax                          0.998065
Profit_Lag_7          Net_Sales_Lag_7              0.997563
Tax                   Gross_Profit                 0.997560
Total_Collected       Gross_Profit                 0.997559
Net_Sales             Gross_Profit                 0.997559
Profit_Lag_1          Net_Sales_Lag_1              0.997558
Net_Sales             Total_Cost                   0.996529
Total_Collected       Total_Cost                   0.996529
Tax                   Total_Cost                   0.996527
Gross_Sales           Gross_Profit      

In [51]:
# Final feature set after correlation-based reduction

selected_features = [
    col for col in feature_candidates
    if col not in ["Gross_Sales", "Tax", "Total_Collected", "Total_Cost"]
]

X_train_final = X_train[selected_features]
X_validation_final = X_validation[selected_features]
X_test_final = X_test[selected_features]

print("Number of selected features:", len(selected_features))
print("Features:")
print(selected_features)

print("\nShapes:")
print("Train:", X_train_final.shape)
print("Validation:", X_validation_final.shape)
print("Test:", X_test_final.shape)

Number of selected features: 26
Features:
['Discounts', 'Net_Sales', 'Gross_Profit', 'Transactions', 'Units', 'Refund_Lines', 'Discounted_Lines', 'Identified_Customer_Lines', 'Unique_SKUs', 'Store_Closed', 'Profit_Margin', 'Discount_Rate', 'Avg_Transaction_Value', 'Units_Per_Transaction', 'Profit_Lag_1', 'Profit_Lag_7', 'Net_Sales_Lag_1', 'Net_Sales_Lag_7', 'Transactions_Lag_1', 'Transactions_Lag_7', 'Profit_Rolling_7D', 'Net_Sales_Rolling_7D', 'Transactions_Rolling_7D', 'Profit_Growth_7D', 'Net_Sales_Growth_7D', 'Transactions_Growth_7D']

Shapes:
Train: (16200, 26)
Validation: (2760, 26)
Test: (2550, 26)


## Save Outputs

Save the final selected-feature datasets and the feature list so the modeling notebook can load them directly without repeating the cleaning/selection steps.

In [52]:
import os

output_dir = "../../data/processed/feature_selection"
os.makedirs(output_dir, exist_ok=True)

# Save feature matrices + targets
X_train_final.assign(**{target: y_train}).to_csv(f"{output_dir}/train_final.csv", index=False)
X_validation_final.assign(**{target: y_validation}).to_csv(f"{output_dir}/validation_final.csv", index=False)
X_test_final.assign(**{target: y_test}).to_csv(f"{output_dir}/test_final.csv", index=False)

# Save the selected feature list + target name
with open(f"{output_dir}/selected_features.json", "w") as f:
    json.dump({"target": target, "selected_features": selected_features}, f, indent=2)

print("Saved train/validation/test sets and selected_features.json to:", output_dir)

Saved train/validation/test sets and selected_features.json to: ../../data/processed/feature_selection
